In [5]:
import pandas as pd

df = pd.read_excel("leiloeiros_associados.xlsx")
df["Site"] = df["Site"].str.replace(r"^http://", "https://", regex=True)

sites_list = df["Site"].to_list()

sites_list[:32]

['https://www.alexandrecostaleiloes.com.br/',
 'https://www.alexandroleiloeiro.com.br',
 'https://www.analucialeiloeira.com.br',
 'https://www.andersonleiloeiro.lel.br',
 'https://www.andrealeiloeira.lel.br',
 'https://br.antonioferreira.lel.br/',
 'https://www.bspleiloes.com.br',
 'https://www.facanhaleiloes.com.br/',
 'https://www.depaulaonline.com.br',
 'https://robertohaddad.com.br/',
 'https://www.edgarcarvalholeiloeiro.com.br/',
 'https://fabianoayuppleiloeiro.com.br/',
 'https://www.portellaleiloes.com.br',
 'https://www.gpleilao.com.br/',
 'https://www.gustavoleiloeiro.lel.br',
 'https://www.leilaobrasil.com.br',
 'https://www.rymerleiloes.com.br',
 'https://www.monizdearagao.leilao.br/',
 'https://www.leiloesja.com.br/',
 'https://www.jvleiloes.lel.br',
 'https://www.brameleiloes.com.br',
 'https://www.schulmannleiloes.com.br',
 'https://www.depaulaonline.com.br',
 'https://www.maiconleiloeiro.com.br',
 'https://www.marcellacalsleiloeira.com.br/',
 'https://www.marioricart.lel

In [3]:
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
from datetime import datetime

@dataclass(frozen=True)
class Selector:
    strategy: str
    value: str

@dataclass
class AuctionRound:
    name: str
    end: datetime | None
    start: datetime | None = None
    initial_bid: float | None = None

@dataclass
class AuctionModel:
    title: str
    description: str
    status: str
    
    rounds: list[AuctionRound] | None =  None
    image_url: str | None = None


class AuctionNavigator(ABC):
    @abstractmethod
    async def has_next_page(self, page) -> bool:
        pass

    @abstractmethod
    async def goto_next_page(self, page):
        pass


class AuctionParser(ABC):
    async def select_section_in_navbar(self, page, navbar_selector):
        if navbar_selector is not None:
            if navbar_selector.strategy == "css":
                await page.locator(navbar_selector.value).click()
            elif navbar_selector.strategy == "text":
                await page.get_by_text(navbar_selector.value).click()

    async def parse_auction(self, card):
        return AuctionModel(
            title=await self._title(card),
            description=await self._description(card),
            status=await self._status(card),
            rounds=await self._rounds(card),
            image_url=await self._image_url(card)
        )
    
    @abstractmethod
    async def get_auction_cards(self, page):
        pass

    @abstractmethod
    async def _title(self, card):
        pass
        
    @abstractmethod
    async def _description(self, card):
        pass

    @abstractmethod
    async def _status(self, card):
        pass

    @abstractmethod
    async def _rounds(self, card):
        pass

    @abstractmethod
    async def _image_url(self, card):
        pass


@dataclass
class SiteConfig:
    auction_parser: AuctionParser
    auction_navigator: AuctionNavigator
    auction_navbar_selector: Selector | None = None
    show_more_selector: Selector | None = None

In [4]:
class AlexandreCostaNavigator(AuctionNavigator):
    async def has_next_page(self, page) -> bool:
        next_button = page.locator(
        ".Anuncio1_seletor:has(.Anuncio1_seletorStr:text-is('>'))"
        )

        return await next_button.count() > 0

    async def goto_next_page(self, page):
        next_button = page.locator(
            ".Anuncio1_seletor:has(.Anuncio1_seletorStr:text-is('>'))"
        )
    
        await next_button.click()
        await page.wait_for_load_state("networkidle")


class AlexandreLeiloeiroNavigator(AuctionNavigator):
    async def has_next_page(self, page) -> bool:
        return False

    async def goto_next_page(self, page):
        return

In [12]:
import re

class AlexandreCostaParser(AuctionParser):        
    async def get_auction_cards(self, page):
        return page.locator(
            "div[id^='divAnuncio']"
        )

    async def _initial_bid(self, card):
        return None

    async def _title(self, card):
        return await card.locator(".Anuncio1_tit").text_content()
        
    async def _description(self, card):
        return await card.locator(".Anuncio1_descr").text_content()

    async def _status(self, card):
        return await card.locator("#btnCons").get_attribute("value")

    async def _rounds(self, card):
        rounds = []
    
        # 2º filho direto do card = bloco dos rounds
        rounds_container = card.locator(":scope > div").nth(1)
    
        # Procurar todos os elementos que representam um round
        # Eles sempre contêm exatamente 2 elementos .Anuncio1_data
        candidates = rounds_container.locator("div:has(.Anuncio1_data)")
    
        count = await candidates.count()
    
        for i in range(count):
            candidate = candidates.nth(i)
    
            data_fields = candidate.locator(".Anuncio1_data")
    
            # Um round válido tem exatamente 2 campos: nome + data
            if await data_fields.count() == 2:
                name = (await data_fields.nth(0).text_content() or "").strip()
                date = (await data_fields.nth(1).text_content() or "").strip()
    
                # Evitar capturar containers intermediários vazios
                if name and date:
                    rounds.append(
                        AuctionRound(
                            name=name,
                            end=date
                        )
                    )
    
        return rounds

    async def _image_url(self, card):
        image = (
            card.locator("div[class^='Anuncio'][class$='_img']")
                .locator("img")
                .first
        )
        return await image.get_attribute("src")

        
class AlexandreLeiloeiroParser(AuctionParser):
    async def get_auction_cards(self, page):
        return page.locator(
            "article[class^='evento-index']"
        )

    async def _initial_bid(self, card):
        return None

    async def _title(self, card):
        return await card.locator("h3").text_content()

    async def _description(self, card):
        return None

    async def _status(self, card):
        return await card.locator(".strong-status").text_content()

    async def _rounds(self, card):
        rounds = []
    
        round_cards = card.locator("ul.cont-datas > li")
    
        for i in range(await round_cards.count()):
            round_card = round_cards.nth(i)
    
            if not await round_card.is_visible():
                continue
    
            name = await round_card.locator(".line-1 strong").text_content()
            start = await round_card.locator(".col-line:not(.cl-2) span").text_content()
            end = await round_card.locator(".col-line.cl-2 span").text_content()
    
            rounds.append(
                AuctionRound(
                    name=(name or "").strip(),
                    start=(start or "").strip(),
                    end=(end or "").strip(),
                )
            )
    
        return rounds

    async def _image_url(self, card):
        image = card.locator(".cont-picture img")
        if await image.count() == 0:
            return None
    
        return await image.first.get_attribute("src")


class AndresRosaCostaParser(AlexandreCostaParser):
    async def _title(self, card):
        return await card.locator(".Anuncio2_faixa").text_content()
    
    async def _initial_bid(self, card):
        return None
    
    async def _description(self, card):
        blocks = card.locator(":scope > div")

        return await blocks.nth(2).locator(
            "div"
        ).first.text_content()

    async def _status(self, card):
        return await card.locator("#btnCons").get_attribute("value")

    async def _rounds(self, card):
        rounds = []

        blocks = card.locator(":scope > div")
    
        for i in range(await blocks.count()):
            text = await blocks.nth(i).inner_text()
    
            if "LEILÃO:" not in text:
                continue
    
            round_data = self._parse_round(text)
    
            if round_data:
                rounds.append(round_data)
    
        return rounds

    def _parse_round(self, text: str) -> AuctionRound | None:
        """
        Exemplo de entrada:
    
        1° LEILÃO: 24/06/2021 - 11:15H
        LANCE INICIAL: 208.429,64
        """
    
        name_match = re.search(
            r"(\d+°\s*LEILÃO)",
            text
        )
    
        end_match = re.search(
            r"LEILÃO:\s*(\d{2}/\d{2}/\d{4})\s*-\s*(\d{2}:\d{2})H",
            text
        )
    
        if not name_match:
            return None
    
        end = None
    
        if end_match:
            date_str = end_match.group(1)
            time_str = end_match.group(2)
    
            end = datetime.strptime(
                f"{date_str} {time_str}",
                "%d/%m/%Y %H:%M"
            )
    
        return AuctionRound(
            name=name_match.group(1),
            end=end
        )

class FacanhaLeiloesParser(AlexandreLeiloeiroParser):
    async def _rounds(self, card):
        rounds = []
    
        round_cards = card.locator(
            ".cont-datas > li"
        )
    
        for i in range(await round_cards.count()):
            round_card = round_cards.nth(i)
    
            name = await round_card.locator(
                ".line-1 strong"
            ).text_content()
    
            date_text = await round_card.locator(
                ".col-line"
            ).nth(0).locator("span").text_content()
    
            bid_text = await round_card.locator(
                ".col-line"
            ).nth(1).locator("span").text_content()
    
            rounds.append(
                AuctionRound(
                    name=name.strip(),
                    end=self._parse_datetime(date_text),
                    initial_bid=self._parse_money(bid_text)
                )
            )
    
        return rounds

    def _parse_datetime(self, value: str):
        if not value:
            return None
    
        return datetime.strptime(
            value.strip(),
            "%d/%m/%Y %H:%M"
        )

    def _parse_money(self, value: str):
        if not value:
            return None
    
        value = (
            value
            .replace("R$", "")
            .replace(".", "")
            .replace(",", ".")
            .strip()
        )
    
        return float(value)


class DePaulaParser(AuctionParser):
    async def get_auction_cards(self, page):
        return page.locator(
            "article"
        )

    async def _title(self, card):
        return await card.locator("h3").text_content()
        
    async def _description(self, card):
        return await card.locator().text_content()

    async def _status(self, card):
        pass

    async def _rounds(self, card):
        pass

    async def _image_url(self, card):
        pass

In [13]:
site_template_dict = {
    "https://www.alexandrecostaleiloes.com.br/": SiteConfig(
        AlexandreCostaParser(),
        AlexandreCostaNavigator(),
        auction_navbar_selector=Selector("css", "div[class='Topo1_mnu_PrincL']"),
        show_more_selector=Selector("css", ".Anuncio1_seletores")
    ),
    "https://www.alexandroleiloeiro.com.br": SiteConfig(
        AlexandreLeiloeiroParser(),
        AlexandreLeiloeiroNavigator()
    ),
    "https://www.analucialeiloeira.com.br": SiteConfig(
        AlexandreCostaParser(),
        AlexandreCostaNavigator()
    ),
    "https://www.andersonleiloeiro.lel.br": SiteConfig(
        AlexandreCostaParser(),
        AlexandreCostaNavigator()
    ),
    "https://www.andrealeiloeira.lel.br": SiteConfig(
        AndresRosaCostaParser(),
        AlexandreCostaNavigator(),
        auction_navbar_selector=Selector("css", "a[class='Anuncio1_seletor_linknaveg']")
    ),
    "https://www.bspleiloes.com.br": SiteConfig(
        AlexandreCostaParser(),
        AlexandreCostaNavigator(),
        show_more_selector=Selector("css", ".Anuncio1_seletores")
    ),
    "https://www.facanhaleiloes.com.br/": SiteConfig(
        FacanhaLeiloesParser(),
        AlexandreLeiloeiroNavigator()
    ),
     "https://www.depaulaonline.com.br": SiteConfig(
        DePaulaParser(),
        AlexandreLeiloeiroNavigator()
    )
}

In [14]:
from playwright.async_api import async_playwright

p = await async_playwright().start()

browser = await p.chromium.launch(headless=False)

page = await browser.new_page()

num_auctions_for_site = 0
total_auctions = 0
for site in sites_list[8:9]:
    url = site.strip()

    site_config = site_template_dict.get(url, None)
    if site_config is None:
        print(f"Site template for {url} is None. Continuing...")
        continue
        
    await page.goto(url)

    parser = site_config.auction_parser
    navigator = site_config.auction_navigator
    await parser.select_section_in_navbar(page, site_config.auction_navbar_selector)

    while True:
        cards = await parser.get_auction_cards(page)
        await cards.first.wait_for(
            state="visible",
            timeout=30000
        )
                
        count = await cards.count()
        for i in range(count):
            card = cards.nth(i)
    
            print(await card.evaluate("el => el.outerHTML"))
            # if i == 1:
            break
            
            auction = await parser.parse_auction(card)
            # print(auction)
            num_auctions_for_site += 1
        if not await navigator.has_next_page(page):
            break

        break
        await navigator.goto_next_page(page)

    print(f"Num auction for site {site}: {num_auctions_for_site}")
    total_auctions += num_auctions_for_site
    num_auctions_for_site = 0
           
    break

    print("\n\n\n")
    # await browser.close()

# print(total_auctions)
# await p.stop()

<article><input id="imoveis" type="checkbox" name="" value="1" class="inp-cbx" style="display: none;"> <label for="imoveis" class="cbx"><span><svg width="13px" height="13px" viewBox="0 0 12 10"><polyline points="1.5 6 4.5 9 10.5 1"></polyline></svg></span> <small>Imóveis</small></label></article>
Num auction for site https://www.depaulaonline.com.br: 0
